In [1]:
import os, json, re, time, random, math, gc, inspect, shutil
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    TrainerCallback,
)

# === CONFIG ===
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

SEED = 42
SAFE_EOS_ID = 50256
MAX_LEN = 256              # V22: down from 320 â†’ even faster training

import pathlib
def first_existing(*paths):
    for p in map(pathlib.Path, paths):
        if p.exists():
            return str(p)
    raise FileNotFoundError("No path found: " + " | ".join(paths))

MODEL_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
)
DATA_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "/kaggle/input/dataset-math",
)

OUTPUT_DIR = "/kaggle/working/gpt2-math-finetuned"
BEST_LOSS_DIR = "/kaggle/working/gpt2-math-finetuned-best-loss"

PROMPT_TEMPLATE = (
    "### Bài toán:\n{q}\n\n"
    "### Yêu cầu:\n"
    "Giải ngắn gọn. Dòng cuối bắt buộc: Đáp án là: <số>.\n\n"
    "### Lời giải:\n"
)

# V22 config â€” MODEL-ONLY, no fuzzy bypass
MAX_TRAIN_SAMPLES = 80000
NUM_EPOCHS = 9999
LR = 1e-3
PER_DEVICE_BS = 16
GRAD_ACCUM = 1
ANSWER_WEIGHT = 3.0
TRAIN_TIME_LIMIT = 165     # V22: up from 160, more training time

MAX_NEW_TOKENS = 192
GEN_BATCH_SIZE = 24
PRUNE_KEEP_LINES = 3       # V22: down from 4 â†’ even shorter sequences

NOTEBOOK_START = time.time()

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")
print(f"MODEL_DIR: {MODEL_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"Effective Batch Size: {PER_DEVICE_BS * GRAD_ACCUM * max(1, torch.cuda.device_count())}")
print(f"MAX_LEN: {MAX_LEN} | PRUNE_KEEP_LINES: {PRUNE_KEEP_LINES}")
print(f"STRATEGY: MODEL-ONLY (no fuzzy match bypass)")


CUDA available: True
GPU: Tesla T4
GPU count: 2
MODEL_DIR: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
DATA_DIR: /kaggle/input/datasets/kimanh2002/dataset-math
Effective Batch Size: 32
MAX_LEN: 256 | PRUNE_KEEP_LINES: 3
STRATEGY: MODEL-ONLY (no fuzzy match bypass)


In [2]:
def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        content = f.read().strip()
    if content.startswith("["):
        return json.loads(content)
    data = []
    for line in content.split("\n"):
        line = line.strip()
        if line:
            data.append(json.loads(line))
    return data


_NUM_RE = re.compile(r"[+-]?\d+(?:[\.,]\d+)?(?:\s*/\s*[+-]?\d+(?:[\.,]\d+)?)?")
_ANS_RE = re.compile(
    r"(?:Đáp\s*án\s*là|Đáp\s*án|Câu\s*trả\s*lời\s*là|Kết\s*quả\s*là|####)\s*[:：]?\s*("
    + _NUM_RE.pattern + r")",
    re.IGNORECASE,
)


def normalize_response(resp):
    resp = (resp or "").strip()
    resp = re.sub(
        r"Câu\s*trả\s*lời\s*(?:là)?\s*[:：]?\s*",
        "Đáp án là: ",
        resp, flags=re.IGNORECASE
    )
    resp = re.sub(
        r"Kết\s*quả\s*(?:là)?\s*[:：]?\s*",
        "Đáp án là: ",
        resp, flags=re.IGNORECASE
    )
    resp = re.sub(
        r"[Đđ]áp\s*án\s*[:：]\s*",
        "Đáp án là: ",
        resp, flags=re.IGNORECASE
    )
    resp = re.sub(r"\n{3,}", "\n\n", resp)
    return resp


def extract_answer_fast(text):
    if not text:
        return None
    matches = list(_ANS_RE.finditer(text))
    if matches:
        return matches[-1].group(1).replace(" ", "").rstrip(".,;")
    nums = _NUM_RE.findall(text)
    return nums[-1].replace(" ", "").rstrip(".,;") if nums else None


def normalize_key(text):
    text = (text or "").lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s\dàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ/%.,+-]", "", text)
    return text


def quality_score(q, r, ans):
    score = 0
    if ans:
        score += 10
    if re.search(r"Đáp\s*án\s*là\s*[:：]", r, re.IGNORECASE):
        score += 8
    if re.fullmatch(_NUM_RE, ans or ""):
        score += 5
    if 20 <= len(q) <= 1000:
        score += 4
    if 20 <= len(r) <= 1600:
        score += 4
    if len(r) <= 1000:
        score += 2
    score -= max(0, len(q) - 1800) // 300
    score -= max(0, len(r) - 2600) // 400
    return score


def select_train_data(records, target=None, seed=42):
    rng = random.Random(seed)
    stats = Counter()
    best = {}
    for rec in records:
        q = (rec.get("query_vi") or "").strip()
        r = normalize_response(rec.get("response_vi", ""))
        if not q or not r:
            stats["empty"] += 1
            continue
        if len(q) > 2600 or len(r) > 4200:
            stats["too_long"] += 1
            continue
        ans = extract_answer_fast(r)
        if ans is None:
            stats["no_numeric_answer"] += 1
            continue
        new_rec = dict(rec)
        new_rec["query_vi"] = q
        new_rec["response_vi"] = r
        new_rec["_answer"] = ans
        key = normalize_key(q)
        sc = quality_score(q, r, ans)
        old = best.get(key)
        if old is None or sc > old[0]:
            best[key] = (sc, new_rec)
        else:
            stats["duplicate_lower_quality"] += 1
    scored = list(best.values())
    if target is None or len(scored) <= target:
        scored.sort(key=lambda x: (-x[0], len(x[1]["query_vi"]), len(x[1]["response_vi"])))
        selected = [x[1] for x in scored]
    else:
        buckets = defaultdict(list)
        for sc, rec in scored:
            buckets[str(rec.get("type") or "__none__")].append((sc, rec))
        selected = []
        total = len(scored)
        for _, items in sorted(buckets.items(), key=lambda kv: -len(kv[1])):
            rng.shuffle(items)
            items.sort(key=lambda x: (-x[0], len(x[1]["query_vi"]), len(x[1]["response_vi"])))
            quota = max(1, round(target * len(items) / total))
            selected.extend([rec for _, rec in items[:quota]])
        if len(selected) > target:
            selected = selected[:target]
        elif len(selected) < target:
            selected_keys = {normalize_key(x["query_vi"]) for x in selected}
            rest = [(sc, rec) for sc, rec in scored if normalize_key(rec["query_vi"]) not in selected_keys]
            rest.sort(key=lambda x: (-x[0], len(x[1]["query_vi"])))
            selected.extend([rec for _, rec in rest[:target-len(selected)]])
    print("[clean] raw:", len(records), "| unique usable:", len(scored), "| selected:", len(selected))
    print("[clean] skipped:", dict(stats))
    return selected


# â”€â”€ V22: Response Pruning (more aggressive: 3 lines) â”€â”€
def prune_response(resp, keep_lines=PRUNE_KEEP_LINES):
    """Keep only last N reasoning lines before the answer line."""
    resp = (resp or "").strip()
    if not resp:
        return resp
    lines = [l for l in resp.split("\n") if l.strip()]
    if len(lines) <= keep_lines + 1:
        return resp
    ans_idx = -1
    for i in range(len(lines) - 1, -1, -1):
        if re.search(r"(?:Đáp\s*án\s*là|####|Câu\s*trả\s*lời|Kết\s*quả\s*là)", lines[i], re.IGNORECASE):
            ans_idx = i
            break
    if ans_idx == -1:
        return "\n".join(lines[-keep_lines:])
    start = max(0, ans_idx - keep_lines)
    return "\n".join(lines[start:ans_idx + 1])


# Load data
train_raw = load_data(os.path.join(DATA_DIR, "train.json"))
valid_path = os.path.join(DATA_DIR, "valid.json")
if not os.path.exists(valid_path):
    valid_path = os.path.join(DATA_DIR, "val.json")
valid_data = load_data(valid_path)

train_data = select_train_data(train_raw, MAX_TRAIN_SAMPLES, SEED)

# Apply Response Pruning
prune_stats = {'pruned': 0, 'kept': 0}
for rec in train_data:
    original = rec.get("response_vi", "")
    pruned = prune_response(original, PRUNE_KEEP_LINES)
    if len(pruned) < len(original):
        prune_stats['pruned'] += 1
    else:
        prune_stats['kept'] += 1
    rec["response_vi_pruned"] = pruned

for rec in valid_data:
    rec["response_vi"] = normalize_response(rec.get("response_vi", ""))

print(f"Train: {len(train_data)} | Valid: {len(valid_data)}")
print(f"Prune stats: {prune_stats}")


[clean] raw: 95400 | unique usable: 57683 | selected: 57683
[clean] skipped: {'duplicate_lower_quality': 37318, 'no_numeric_answer': 41}
Train: 57683 | Valid: 1000
Prune stats: {'pruned': 223, 'kept': 57460}


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID
tokenizer.model_max_length = MAX_LEN


def strip_final_answer_line(resp):
    resp = (resp or "").strip()
    resp = re.sub(
        r"(\n|\s)*(?:Đáp\s*án\s*là|Đáp\s*án|Câu\s*trả\s*lời\s*là|Kết\s*quả\s*là|####)\s*[:：]?\s*"
        + _NUM_RE.pattern + r"\s*\.?\s*$",
        "",
        resp,
        flags=re.IGNORECASE,
    ).strip()
    return resp


def make_target_response(resp, ans):
    body = strip_final_answer_line(normalize_response(resp))
    final = f"Đáp án là: {ans}" if ans is not None else ""
    if body:
        return body.rstrip() + "\n" + final
    return final


class MathSFTDataset(Dataset):
    """V22: Uses PRUNED responses, no fuzzy match â€” pure model training."""

    def __init__(self, records, tokenizer, max_len):
        self.items = []
        skipped_empty = 0
        vocab_max = min(len(tokenizer) - 1, SAFE_EOS_ID)
        lens = []

        for rec in records:
            q = (rec.get("query_vi", "") or "").strip()
            ans = rec.get("_answer") or extract_answer_fast(rec.get("response_vi", ""))
            if not q or not ans:
                skipped_empty += 1
                continue

            resp_text = rec.get("response_vi_pruned", rec.get("response_vi", ""))
            target = make_target_response(resp_text, ans)
            if not target.strip():
                skipped_empty += 1
                continue

            prompt = PROMPT_TEMPLATE.format(q=q)
            p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]

            body = strip_final_answer_line(resp_text)
            final_line = f"\nĐáp án là: {ans}"
            body_ids = tokenizer(body.strip(), add_special_tokens=False)["input_ids"] if body.strip() else []
            ans_ids = tokenizer(final_line, add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]

            if len(ans_ids) >= max_len // 2:
                ans_ids = ans_ids[-max(8, max_len // 3):]

            if len(p_ids) + len(body_ids) + len(ans_ids) > max_len:
                min_answer_budget = len(ans_ids)
                if len(p_ids) + min_answer_budget >= max_len:
                    keep_prompt = max_len - min_answer_budget
                    p_ids = p_ids[-keep_prompt:] if keep_prompt > 0 else []
                    body_ids = []
                else:
                    body_budget = max_len - len(p_ids) - min_answer_budget
                    body_ids = body_ids[:max(0, body_budget)]

            ids = (p_ids + body_ids + ans_ids)[:max_len]
            labels = ([-100] * len(p_ids) + body_ids + ans_ids)[:max_len]
            answer_mask = ([0] * len(p_ids) + [0] * len(body_ids) + [1] * len(ans_ids))[:max_len]
            attention_mask = [1] * len(ids)

            if all(x == -100 for x in labels):
                skipped_empty += 1
                continue

            ids = [min(max(int(t), 0), vocab_max) for t in ids]
            labels = [(-100 if t == -100 else min(max(int(t), 0), vocab_max)) for t in labels]

            self.items.append({
                "input_ids": ids,
                "attention_mask": attention_mask,
                "labels": labels,
                "answer_mask": answer_mask,
            })
            lens.append(len(ids))

        if skipped_empty:
            print(f"  Skipped {skipped_empty} empty/no-answer samples")
        if lens:
            print(f"  Built {len(self.items)} examples | avg_len={sum(lens)/len(lens):.1f} | max_len={max(lens)}")
        else:
            print("  Built 0 examples")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(item["labels"], dtype=torch.long),
            "answer_mask": torch.tensor(item["answer_mask"], dtype=torch.long),
        }


class PadCollator:
    def __init__(self, pad_id):
        self.pad_id = pad_id
    def __call__(self, batch):
        max_len = max(len(b["input_ids"]) for b in batch)
        if max_len % 8 != 0:
            max_len += 8 - (max_len % 8)
        input_ids, attn, labels, masks = [], [], [], []
        for b in batch:
            pad_n = max_len - len(b["input_ids"])
            input_ids.append(torch.cat([b["input_ids"], torch.full((pad_n,), self.pad_id, dtype=torch.long)]))
            attn.append(torch.cat([b["attention_mask"], torch.zeros(pad_n, dtype=torch.long)]))
            labels.append(torch.cat([b["labels"], torch.full((pad_n,), -100, dtype=torch.long)]))
            masks.append(torch.cat([b["answer_mask"], torch.zeros(pad_n, dtype=torch.long)]))
        return {
            "input_ids": torch.stack(input_ids),
            "attention_mask": torch.stack(attn),
            "labels": torch.stack(labels),
            "answer_mask": torch.stack(masks),
        }


print("Building train dataset with PRUNED responses...")
train_ds = MathSFTDataset(train_data, tokenizer, MAX_LEN)
print(f"Validation records available: {len(valid_data)}")


Token indices sequence length is longer than the specified maximum sequence length for this model (277 > 256). Running this sequence through the model will result in indexing errors


Building train dataset with PRUNED responses...
  Built 57683 examples | avg_len=222.4 | max_len=256
Validation records available: 1000


In [4]:
class GPT2ForMath(GPT2LMHeadModel):
    def forward(self, input_ids=None, attention_mask=None, labels=None, answer_mask=None, **kwargs):
        kwargs.pop('num_items_in_batch', None)
        outputs = super().forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs
        )
        if labels is not None:
            shift_logits = outputs.logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fct = nn.CrossEntropyLoss(reduction='none', ignore_index=-100)
            flat_loss = loss_fct(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            ).view(shift_labels.size())
            if answer_mask is not None:
                shift_mask = answer_mask[..., 1:].contiguous()
                token_w = torch.where(
                    shift_mask == 1,
                    torch.tensor(ANSWER_WEIGHT, device=flat_loss.device, dtype=flat_loss.dtype),
                    torch.tensor(1.0, device=flat_loss.device, dtype=flat_loss.dtype)
                )
            else:
                token_w = torch.ones_like(flat_loss)
            valid = (shift_labels != -100).float()
            loss = (flat_loss * token_w * valid).sum() / (token_w * valid).sum().clamp(min=1e-9)
            dummy_logits = torch.zeros(
                (input_ids.size(0), 1, 1),
                device=outputs.logits.device,
                dtype=outputs.logits.dtype
            )
            return {'loss': loss, 'logits': dummy_logits}
        return outputs

GPT2ForMath.__module__ = 'transformers.models.gpt2.modeling_gpt2'


class TimeLimitCallback(TrainerCallback):
    def __init__(self, time_limit_mins, notebook_start):
        self.time_limit_secs = time_limit_mins * 60
        self.notebook_start = notebook_start
    def on_step_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.notebook_start
        if elapsed > self.time_limit_secs:
            print(f"\nTime limit reached ({elapsed/60:.1f} mins). Stopping!")
            control.should_training_stop = True
        return control


class BestTrainLossCallback(TrainerCallback):
    """Save the checkpoint with the lowest logged training loss."""
    def __init__(self, save_dir):
        self.save_dir = save_dir
        self.best_loss = float("inf")
        self.best_step = -1

    def on_log(self, args, state, control, logs=None, model=None, **kwargs):
        if not logs or "loss" not in logs or model is None:
            return control
        try:
            loss = float(logs["loss"])
        except Exception:
            return control
        if not math.isfinite(loss):
            return control
        if loss < self.best_loss and getattr(state, "is_world_process_zero", True):
            self.best_loss = loss
            self.best_step = int(state.global_step)
            if os.path.exists(self.save_dir):
                shutil.rmtree(self.save_dir)
            os.makedirs(self.save_dir, exist_ok=True)
            model.save_pretrained(self.save_dir)
            tokenizer.save_pretrained(self.save_dir)
            info = {"best_train_loss": self.best_loss, "best_step": self.best_step}
            with open(os.path.join(self.save_dir, "best_train_loss.json"), "w", encoding="utf-8") as f:
                json.dump(info, f, ensure_ascii=False, indent=2)
            print(f"\n[best-loss] step={self.best_step} loss={self.best_loss:.6f} saved to {self.save_dir}")
        return control

def build_training_args():
    kwargs = dict(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BS,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        weight_decay=0.0,
        warmup_steps=100,
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        save_strategy="no",
        load_best_model_at_end=False,
        seed=SEED,
        dataloader_num_workers=2,
        dataloader_pin_memory=torch.cuda.is_available(),
        remove_unused_columns=False,
        prediction_loss_only=True,
        max_grad_norm=1.0,
        report_to="none",
    )
    sig = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "no"
    elif "evaluation_strategy" in sig.parameters:
        kwargs["evaluation_strategy"] = "no"
    if "group_by_length" in sig.parameters:
        kwargs["group_by_length"] = True
    if "optim" in sig.parameters:
        kwargs["optim"] = "adamw_torch"
    return TrainingArguments(**kwargs)


if os.path.exists(BEST_LOSS_DIR):
    shutil.rmtree(BEST_LOSS_DIR)

model = GPT2ForMath.from_pretrained(MODEL_DIR, local_files_only=True)
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID
model.config.use_cache = False
if len(tokenizer) != model.get_input_embeddings().weight.shape[0]:
    model.resize_token_embeddings(len(tokenizer))

eff_batch = PER_DEVICE_BS * GRAD_ACCUM * max(1, torch.cuda.device_count())
steps_per_pass = math.ceil(len(train_ds) / eff_batch)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print(f"bs={PER_DEVICE_BS} | effective_bs={eff_batch} | steps/pass~{steps_per_pass}")

training_args = build_training_args()
best_loss_callback = BestTrainLossCallback(BEST_LOSS_DIR)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=PadCollator(SAFE_EOS_ID),
    callbacks=[TimeLimitCallback(TRAIN_TIME_LIMIT, NOTEBOOK_START), best_loss_callback],
)

t0 = time.time()
try:
    trainer.train()
except RuntimeError as e:
    msg = str(e).lower()
    if "out of memory" not in msg and "cuda" not in msg:
        raise
    print("\n[warn] CUDA OOM. Retrying with smaller batch size.")
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    if os.path.exists(BEST_LOSS_DIR):
        shutil.rmtree(BEST_LOSS_DIR)

    model = GPT2ForMath.from_pretrained(MODEL_DIR, local_files_only=True)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.config.use_cache = False
    if len(tokenizer) != model.get_input_embeddings().weight.shape[0]:
        model.resize_token_embeddings(len(tokenizer))

    training_args = build_training_args()
    training_args.per_device_train_batch_size = 8
    training_args.gradient_accumulation_steps = 2
    best_loss_callback = BestTrainLossCallback(BEST_LOSS_DIR)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        data_collator=PadCollator(SAFE_EOS_ID),
        callbacks=[TimeLimitCallback(TRAIN_TIME_LIMIT, NOTEBOOK_START), best_loss_callback],
    )
    trainer.train()

train_time = time.time() - t0
print(f"\nTraining done in {train_time/60:.1f} min")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Final model saved to {OUTPUT_DIR}")

if not os.path.exists(BEST_LOSS_DIR):
    trainer.save_model(BEST_LOSS_DIR)
    tokenizer.save_pretrained(BEST_LOSS_DIR)
    print(f"No logged loss checkpoint found; copied final model to {BEST_LOSS_DIR}")
else:
    print(f"Best logged train loss: step={best_loss_callback.best_step} loss={best_loss_callback.best_loss:.6f}")
    print(f"Best-loss model saved to {BEST_LOSS_DIR}")

del trainer, model
gc.collect()
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForMath LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model params: 124,440,576
bs=16 | effective_bs=32 | steps/pass~1803


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,3.887674
100,2.754957
150,2.437476
200,2.339185
250,2.189244
300,2.066083
350,1.973558
400,1.932284
450,1.891926
500,1.867673


...


[best-loss] step=7300 loss=0.499786 saved to /kaggle/working/gpt2-math-finetuned-best-loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


[best-loss] step=7350 loss=0.497231 saved to /kaggle/working/gpt2-math-finetuned-best-loss

Time limit reached (165.0 mins). Stopping!

Training done in 163.5 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to /kaggle/working/gpt2-math-finetuned
Best logged train loss: step=7350 loss=0.497231
Best-loss model saved to /kaggle/working/gpt2-math-finetuned-best-loss


In [5]:
def extract_answer(text):
    if not text:
        return None
    def clean(x):
        if x is None: return None
        x = str(x).strip().rstrip(".。、,;")
        nums = re.findall(r"[+-]?\d+(?:[\.,]\d+)?(?:\s*/\s*[+-]?\d+(?:[\.,]\d+)?)?", x)
        return nums[-1].replace(" ", "") if nums else x
    patterns = [
        r"[Đđ]áp\s*án\s*(?:là)?\s*[:：]?\s*([+-]?\d[\d.,]*(?:\s*/\s*[+-]?\d[\d.,]*)?)",
        r"[Vv]ậy\s*(?:[Đđ]áp\s*án|[Kk]ết\s*quả|[Ss]ố\s*\w+)?\s*(?:là)?\s*[:：]?\s*([+-]?\d[\d.,]*(?:\s*/\s*[+-]?\d[\d.,]*)?)",
        r"[Cc]âu\s*trả\s*lời\s*(?:là)?\s*[:：]?\s*([+-]?\d[\d.,]*(?:\s*/\s*[+-]?\d[\d.,]*)?)",
        r"[Kk]ết\s*quả\s*(?:là)?\s*[:：]?\s*([+-]?\d[\d.,]*(?:\s*/\s*[+-]?\d[\d.,]*)?)",
        r"####\s*([+-]?\d[\d.,]*(?:\s*/\s*[+-]?\d[\d.,]*)?)",
    ]
    for pattern in patterns:
        matches = list(re.finditer(pattern, text, flags=re.IGNORECASE))
        if matches:
            return clean(matches[-1].group(1))
    m = re.search(r"\\boxed\{([^}]+)\}", text)
    if m:
        val = clean(m.group(1))
        if val: return val
    nums = re.findall(r"[+-]?\d+(?:[\.,]\d+)?(?:\s*/\s*[+-]?\d+(?:[\.,]\d+)?)?", text)
    return nums[-1].replace(" ", "") if nums else None


def parse_number(s):
    if s is None: return None
    s = str(s).strip().rstrip(".")
    if not s: return None
    if "/" in s:
        try:
            a, b = s.split("/", 1)
            return float(a.replace(",",".")) / float(b.replace(",","."))
        except Exception: return None
    if re.fullmatch(r"-?\d{1,3}(\.\d{3})+", s):
        s = s.replace(".", "")
    elif re.fullmatch(r"-?\d{1,3}(,\d{3})+", s):
        s = s.replace(",", "")
    else:
        s = s.replace(",", ".")
    try:
        val = float(s)
        return val if math.isfinite(val) else None
    except ValueError: return None


def clean_model_output(text):
    text = (text or "").strip()
    if not text:
        return "Đáp án là: 0"
    text = re.split(r"\n\s*###\s*Bài toán\s*:", text, maxsplit=1)[0].strip()
    text = re.split(r"\n\s*###\s*Yêu cầu\s*:", text, maxsplit=1)[0].strip()
    text = re.split(r"\n\s*###\s*Lời giải\s*:", text, maxsplit=1)[0].strip()
    lines = text.split("\n")
    if len(lines) > 5:
        line_counts = Counter(l.strip() for l in lines if l.strip())
        repeated = {l for l, c in line_counts.items() if c >= 3 and len(l) > 10}
        if repeated:
            seen_repeat = {}
            clean_lines = []
            for l in lines:
                ls = l.strip()
                if ls in repeated:
                    seen_repeat[ls] = seen_repeat.get(ls, 0) + 1
                    if seen_repeat[ls] <= 2:
                        clean_lines.append(l)
                else:
                    clean_lines.append(l)
            text = "\n".join(clean_lines).strip()
    ans = extract_answer(text)
    if ans is not None and not re.search(r"Đáp\s*án\s*là\s*[:：]?", text[-180:], flags=re.IGNORECASE):
        text = text.rstrip(". \\n") + f"\nĐáp án là: {ans}"
    return text


@torch.no_grad()
def generate_answers_batch(model, tokenizer, queries, device, batch_size=GEN_BATCH_SIZE):
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    all_outputs = []
    for start in range(0, len(queries), batch_size):
        batch_q = queries[start:start+batch_size]
        prompts = [PROMPT_TEMPLATE.format(q=(q or "").strip()) for q in batch_q]
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
        ).to(device)
        gen_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            use_cache=True,
            pad_token_id=SAFE_EOS_ID,
            eos_token_id=SAFE_EOS_ID,
        )
        plen = inputs["input_ids"].shape[1]
        for j in range(len(batch_q)):
            text = tokenizer.decode(gen_ids[j][plen:], skip_special_tokens=True)
            all_outputs.append(clean_model_output(text))
        if (start // batch_size) % 5 == 0:
            print(f"  Generated {min(start+batch_size, len(queries))}/{len(queries)}...")

    print(f"All {len(queries)} queries processed by MODEL (no bypass)")
    return all_outputs


In [6]:
gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def score_one(pred_num, gold_num):
    if pred_num is None or gold_num is None:
        return 0
    rel_err = abs(pred_num - gold_num) / max(1, abs(gold_num))
    if rel_err <= 0.01:
        return 10
    if rel_err <= 0.10:
        return 5
    if rel_err <= 0.50:
        return 1
    return 0


def evaluate_model_dir(model_dir, model_label):
    print(f"\n{'='*60}")
    print(f"Evaluating candidate: {model_label}")
    print(f"Model dir: {model_dir}")
    print(f"{'='*60}")

    gc.collect()
    torch.cuda.empty_cache()

    model = GPT2LMHeadModel.from_pretrained(model_dir, local_files_only=True)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.config.use_cache = True
    model.to(device)
    model.eval()

    queries = [rec.get("query_vi", "") for rec in valid_data]

    t0 = time.time()
    candidate_outputs = generate_answers_batch(
        model,
        tokenizer,
        queries,
        device,
        batch_size=GEN_BATCH_SIZE,
    )
    candidate_infer_time = time.time() - t0

    candidate_results = []
    for i, (rec, output) in enumerate(zip(valid_data, candidate_outputs)):
        gold_ans = extract_answer(rec.get("response_vi", ""))
        pred_ans = extract_answer(output)

        gold_num = parse_number(gold_ans)
        pred_num = parse_number(pred_ans)

        sc = score_one(pred_num, gold_num)

        candidate_results.append({
            "id": i,
            "type": rec.get("type", ""),
            "gold": gold_ans,
            "pred": pred_ans,
            "gold_num": gold_num,
            "pred_num": pred_num,
            "score": sc,
        })

        if i < 3:
            print(f"\n[Sample {i} | {model_label}]")
            print(f"  Q: {rec.get('query_vi', '')[:120]}...")
            print(f"  Model: {output[:200]}")
            print(f"  Gold: {gold_ans} | Pred: {pred_ans} | Score: {sc}")

    candidate_total = sum(r["score"] for r in candidate_results)
    candidate_n = len(candidate_results)
    candidate_exact = sum(1 for r in candidate_results if r["score"] == 10)
    candidate_close = sum(1 for r in candidate_results if r["score"] >= 5)
    candidate_partial = sum(1 for r in candidate_results if r["score"] >= 1)
    candidate_failed = sum(1 for r in candidate_results if r["score"] == 0)

    print(f"\nResult for {model_label}:")
    print(f"  Samples:  {candidate_n}")
    print(f"  Score:    {candidate_total}/{10*candidate_n}  (avg {candidate_total/candidate_n:.2f}/10)")
    print(f"  Exact:    {candidate_exact}  ({100*candidate_exact/candidate_n:.1f}%)")
    print(f"  Close:    {candidate_close}  ({100*candidate_close/candidate_n:.1f}%)")
    print(f"  Partial:  {candidate_partial}  ({100*candidate_partial/candidate_n:.1f}%)")
    print(f"  Failed:   {candidate_failed}  ({100*candidate_failed/candidate_n:.1f}%)")
    print(f"  Infer:    {candidate_infer_time/60:.1f} min")

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "label": model_label,
        "model_dir": model_dir,
        "outputs": candidate_outputs,
        "results": candidate_results,
        "total": candidate_total,
        "n": candidate_n,
        "exact": candidate_exact,
        "close": candidate_close,
        "partial": candidate_partial,
        "failed": candidate_failed,
        "infer_time": candidate_infer_time,
    }


# ============================================================
# Evaluate both final model and best-loss model
# ============================================================
model_candidates = []

if os.path.exists(OUTPUT_DIR):
    model_candidates.append(("FINAL", OUTPUT_DIR))

if os.path.exists(BEST_LOSS_DIR) and os.path.abspath(BEST_LOSS_DIR) != os.path.abspath(OUTPUT_DIR):
    model_candidates.append(("BEST_TRAIN_LOSS", BEST_LOSS_DIR))

if not model_candidates:
    raise FileNotFoundError("No model candidate found. OUTPUT_DIR and BEST_LOSS_DIR do not exist.")

all_candidate_scores = []

for label, model_dir in model_candidates:
    try:
        summary = evaluate_model_dir(model_dir, label)
        all_candidate_scores.append(summary)
    except Exception as e:
        print(f"[warn] Failed to evaluate {label} at {model_dir}: {repr(e)}")

if not all_candidate_scores:
    raise RuntimeError("All model candidates failed during evaluation.")


# ============================================================
# Select best model by valid raw score
# Tie-break: exact > close > partial > FINAL
# ============================================================
def selection_key(x):
    final_bonus = 1 if x["label"] == "FINAL" else 0
    return (
        x["total"],
        x["exact"],
        x["close"],
        x["partial"],
        final_bonus,
    )


best_summary = max(all_candidate_scores, key=selection_key)

SELECTED_MODEL_LABEL = best_summary["label"]
SELECTED_MODEL_DIR = best_summary["model_dir"]

outputs = best_summary["outputs"]
results = best_summary["results"]

total = best_summary["total"]
n = best_summary["n"]
exact = best_summary["exact"]
close = best_summary["close"]
partial = best_summary["partial"]
failed = best_summary["failed"]
infer_time = sum(x["infer_time"] for x in all_candidate_scores)

print(f"\n{'='*70}")
print("MODEL SELECTION SUMMARY")
print(f"{'='*70}")

for s in all_candidate_scores:
    mark = " <-- SELECTED" if s["model_dir"] == SELECTED_MODEL_DIR else ""
    print(
        f"{s['label']:16s} | "
        f"score={s['total']}/{10*s['n']} "
        f"avg={s['total']/s['n']:.2f}/10 "
        f"exact={s['exact']} "
        f"close={s['close']} "
        f"partial={s['partial']}"
        f"{mark}"
    )

print(f"\nSelected model: {SELECTED_MODEL_LABEL}")
print(f"Selected dir:   {SELECTED_MODEL_DIR}")

print(f"\n{'='*50}")
print("  FINAL VALIDATION RESULTS - SELECTED MODEL")
print(f"{'='*50}")
print(f"  Samples:  {n}")
print(f"  Score:    {total}/{10*n}  (avg {total/n:.2f}/10)")
print(f"  Exact:    {exact}  ({100*exact/n:.1f}%)")
print(f"  Close:    {close}  ({100*close/n:.1f}%)")
print(f"  Partial:  {partial}  ({100*partial/n:.1f}%)")
print(f"  Failed:   {failed}  ({100*failed/n:.1f}%)")
print(f"  Train:    {train_time/60:.1f} min")
print(f"  Infer:    {infer_time/60:.1f} min")
print(f"  AnswerWt: {ANSWER_WEIGHT}")
print(f"  ModelDir: {SELECTED_MODEL_DIR}")
print(f"  TotalNB:  {(time.time()-NOTEBOOK_START)/60:.1f} min")
print("  Strategy: MODEL-ONLY + SELECT BY VALID SCORE")

type_scores = defaultdict(list)
for r in results:
    type_scores[r["type"]].append(r["score"])

print("\nPer-type:")
for t in sorted(type_scores):
    s = type_scores[t]
    print(f"  {t:20s}: avg={sum(s)/len(s):.2f}  n={len(s)}")



Evaluating candidate: FINAL
Model dir: /kaggle/working/gpt2-math-finetuned


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  Generated 24/1000...
  Generated 144/1000...
  Generated 264/1000...
  Generated 384/1000...
  Generated 504/1000...
  Generated 624/1000...
  Generated 744/1000...
  Generated 864/1000...
  Generated 984/1000...
All 1000 queries processed by MODEL (no bypass)

[Sample 0 | FINAL]
  Q: Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về ph...
  Model: Ở lượt đầu tiên, Susan tiến về phía trước 8 ô, nên bây giờ cô ấy ở 8 ô. Ở lượt thứ hai, cô ấy tiến về phía trước 2 ô, nhưng bị lùi lại 5 ô, nên bây giờ cô ấy ở 8 - 2 = 5 ô. Ở lượt thứ ba, cô ấy tiến v
  Gold: 37 | Pred: 48 | Score: 1

[Sample 1 | FINAL]
  Q: Nếu $\angle PQR = \angle PRQ$, và độ dài của QR và PR lần lượt là 5 và 7 thì chu vi của tam giác PQR là bao nhiêu?...
  Model: Vì $\angle PQR = \angle PRQ$ nên ta biết $\angle PQR = \angle PRQ = \angle PRQ$. Vì $\angle PQR = \angle PRQ$ nên ta biết $\angle PRQ = \angle PRQ$. Vì $\angle PQR = \angle PRQ$ nên ta biết $\a

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  Generated 24/1000...
  Generated 144/1000...
  Generated 264/1000...
  Generated 384/1000...
  Generated 504/1000...
  Generated 624/1000...
  Generated 744/1000...
  Generated 864/1000...
  Generated 984/1000...
All 1000 queries processed by MODEL (no bypass)

[Sample 0 | BEST_TRAIN_LOSS]
  Q: Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về ph...
  Model: Susan tiến về phía trước 8 ô nên cô ấy di chuyển 8 ô về phía trước. Ở lượt thứ 2, cô ấy di chuyển 2 ô nhưng bị lùi lại 5 ô, do đó cô ấy di chuyển 2 - 5 = -3 ô. Ở lượt thứ 3, cô ấy di chuyển 6 ô nhưng 
  Gold: 37 | Pred: 5 | Score: 0

[Sample 1 | BEST_TRAIN_LOSS]
  Q: Nếu $\angle PQR = \angle PRQ$, và độ dài của QR và PR lần lượt là 5 và 7 thì chu vi của tam giác PQR là bao nhiêu?...
  Model: Vì $\angle PQR = \angle PRQ$ nên ta biết tam giác PQR là tam giác cân. Do đó, PQ = PR = 7. Chu vi tam giác PQR là PQ + PR + PR = 7 + 5 + 7 = 20. Vì vậy, chu vi của tam giác P

In [7]:
# ═══════════════════════════════════════════════════════════
# Phase 1: Save valid_output.json + valid_report.json
# ═══════════════════════════════════════════════════════════

# --- valid_output.json ---
valid_output_records = []
for i, (rec, output) in enumerate(zip(valid_data, outputs)):
    valid_output_records.append({
        "id": rec.get("id", i),
        "query_vi": rec.get("query_vi", ""),
        "type": rec.get("type", ""),
        "model_output": output,
    })

with open("/kaggle/working/valid_output.json", "w", encoding="utf-8") as f:
    json.dump(valid_output_records, f, ensure_ascii=False, indent=2)
print(f"Wrote valid_output.json ({len(valid_output_records)} records)")

# --- valid_report.json ---
type_scores_report = defaultdict(list)
for r in results:
    type_scores_report[r["type"]].append(r["score"])

score_by_type = {}
for t in sorted(type_scores_report):
    s = type_scores_report[t]
    score_by_type[t] = {
        "n": len(s),
        "mean_score": sum(s) / len(s),
        "exact_count": sum(1 for x in s if x == 10),
        "raw_score": sum(s),
    }

candidate_scores_report = []
for s in all_candidate_scores:
    candidate_scores_report.append({
        "label": s["label"],
        "model_dir": s["model_dir"],
        "raw_score": s["total"],
        "max_raw_score": 10 * s["n"],
        "score_10": s["total"] / s["n"] if s["n"] else 0,
        "exact_count": s["exact"],
        "close_count": s["close"],
        "partial_count": s["partial"],
        "failed_count": s["failed"],
        "infer_time_min": s["infer_time"] / 60,
    })

valid_report = {
    "selected_model_label": SELECTED_MODEL_LABEL,
    "selected_model_dir": SELECTED_MODEL_DIR,
    "selection_metric": "valid_raw_score",
    "candidate_scores": candidate_scores_report,
    "n": n,
    "raw_score": total,
    "max_raw_score": 10 * n,
    "score_10": total / n if n else 0,
    "score_pct": total / (10 * n) if n else 0,
    "exact_count": exact,
    "close_count": close,
    "partial_count": partial,
    "failed_count": failed,
    "buckets": {
        "10": exact,
        "5": sum(1 for r in results if r["score"] == 5),
        "1": sum(1 for r in results if r["score"] == 1),
        "0": failed,
    },
    "score_by_type": score_by_type,
    "rows": results,
}

with open("/kaggle/working/valid_report.json", "w", encoding="utf-8") as f:
    json.dump(valid_report, f, ensure_ascii=False, indent=2)
print("Wrote valid_report.json")
print(f"Selected model for valid_output.json: {SELECTED_MODEL_LABEL} -> {SELECTED_MODEL_DIR}")


Wrote valid_output.json (1000 records)
Wrote valid_report.json
Selected model for valid_output.json: BEST_TRAIN_LOSS -> /kaggle/working/gpt2-math-finetuned-best-loss


In [8]:

import os
import json
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

FINAL_MODEL_DIR = "/kaggle/working/gpt2-math-finetuned"
BEST_LOSS_MODEL_DIR = "/kaggle/working/gpt2-math-finetuned-best-loss"

FINAL_OUTPUT_PATH = "/kaggle/working/test_predictions.json"
BEST_LOSS_OUTPUT_PATH = "/kaggle/working/test_predictions_best_loss.json"


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_inference_model(model_dir):
    print("=" * 60)
    print("Loading model from:", model_dir)
    print("=" * 60)

    tokenizer = AutoTokenizer.from_pretrained(
        model_dir,
        local_files_only=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        local_files_only=True
    ).to(device)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.eos_token_id
    model.config.use_cache = True

    model.eval()
    return model, tokenizer


def run_test_inference(model_dir, output_path):
    clear_gpu()

    model, tokenizer = load_inference_model(model_dir)

    test_data = load_data(os.path.join(DATA_DIR, "test.json"))
    queries_test = [rec.get("query_vi", "") for rec in test_data]

    print("Number of test samples:", len(test_data))
    print("Generating answers...")

    outputs_test = generate_answers_batch(
        model,
        tokenizer,
        queries_test,
        device
    )

    test_preds = [
        {
            "id": rec.get("id", i),
            "query_vi": rec.get("query_vi", ""),
            'type': rec.get('type', ''),
            "model_output": out
        }
        for i, (rec, out) in enumerate(zip(test_data, outputs_test))
    ]

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(test_preds, f, ensure_ascii=False, indent=2)

    print("Wrote:", output_path)
    print("Total predictions:", len(test_preds))

    del model
    del tokenizer
    clear_gpu()


# ============================================================
# 1. Inference with FINAL model
# ============================================================

run_test_inference(
    model_dir=FINAL_MODEL_DIR,
    output_path=FINAL_OUTPUT_PATH
)


# ============================================================
# 2. Inference with BEST-LOSS model
# ============================================================

run_test_inference(
    model_dir=BEST_LOSS_MODEL_DIR,
    output_path=BEST_LOSS_OUTPUT_PATH
)


# ============================================================
# 3. Check output files
# ============================================================

print("\n" + "=" * 60)
print("OUTPUT FILES")
print("=" * 60)

for p in [FINAL_OUTPUT_PATH, BEST_LOSS_OUTPUT_PATH]:
    print(p, "exists=", os.path.exists(p), "size=", os.path.getsize(p) if os.path.exists(p) else None)

Loading model from: /kaggle/working/gpt2-math-finetuned


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Number of test samples: 1000
Generating answers...
  Generated 24/1000...
  Generated 144/1000...
  Generated 264/1000...
  Generated 384/1000...
  Generated 504/1000...
  Generated 624/1000...
  Generated 744/1000...
  Generated 864/1000...
  Generated 984/1000...
All 1000 queries processed by MODEL (no bypass)
Wrote: /kaggle/working/test_predictions.json
Total predictions: 1000
Loading model from: /kaggle/working/gpt2-math-finetuned-best-loss


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Number of test samples: 1000
Generating answers...
  Generated 24/1000...
  Generated 144/1000...
  Generated 264/1000...
  Generated 384/1000...
  Generated 504/1000...
  Generated 624/1000...
  Generated 744/1000...
  Generated 864/1000...
  Generated 984/1000...
All 1000 queries processed by MODEL (no bypass)
Wrote: /kaggle/working/test_predictions_best_loss.json
Total predictions: 1000

OUTPUT FILES
/kaggle/working/test_predictions.json exists= True size= 830258
/kaggle/working/test_predictions_best_loss.json exists= True size= 757594
